# Tutorial: Finding, Accessing, and Comparing Data in DKRZ Waterpark

## Introduction

Waterpark is an interactive data exploration environment developed by DKRZ that simplifies access to climate and Earth system datasets. Rather than manually locating files on the HPC filesystem, users can browse datasets, inspect metadata, preview variables, and launch analyses.
A typical workflow consists of following steps:

- Find relevant datasets
- Access the data
- Process the data
- Compare datasets

The waterpark landing page allows you to explore datasets using metadata rather than filenames. If your datasets are published through a STAC (SpatioTemporal Asset Catalog) interface, the workflow becomes much more flexible because you discover datasets from metadata instead of browsing directories.


In [1]:
from pystac_client import Client
import xarray as xr
import fsspec

## Workflow

### Connect to the STAC Catalog

In [2]:
catalog = Client.open("https://freva.dkrz.de/api/freva-nextgen/stacapi/product/?visible_collections=cmip6%2Cdyamond%2Ceerie%2Cicdc%2Cicon-dream%2Cnextgems%2Corchestra")

### Find the Datasets

In [3]:
nextgems_search = catalog.search(
    collections=["nextgems"],
    datetime="2020-01-01/2020-12-31"
)

nextgems_item = next(nextgems_search.items())


In [4]:
nextgems_item

<Item id=1870346051585572864>

In [5]:
nextgems_item.assets["zarr-access"].href.split("=")[1]

's3://nextgems/timeChunks/ngc4008_PT15M_2.zarr'

**Link in STAC does not match what is needed for opening dataset!**

In [6]:
ds_nextgems = xr.open_zarr(fsspec.get_mapper("s3:///nextgems/healpix/ngc4008/P1D/level_2.zarr", endpoint_url="https://s3.waterpark.dkrz.de", anon=True))

In [7]:
ds_era5 = xr.open_zarr(fsspec.get_mapper("s3://reanalysis/healpix/era5/P1D/level_2.zarr", endpoint_url="https://s3.waterpark.dkrz.de", anon=True))

### Select Variable

In [8]:
tas_nextgems = ds_nextgems["tas"]

In [9]:
tas_era5 = ds_era5["tas"]

### Match Time Period

In [10]:
tas_nextgems.time

<xarray.DataArray 'time' (time: 10958)> Size: 88kB
[10958 values with dtype=int64]
Dimensions without coordinates: time

In [11]:
tas_era5.time

<xarray.DataArray 'time' (time: 31452)> Size: 252kB
array(['1940-01-01T12:00:00.000000000', '1940-01-02T12:00:00.000000000',
       '1940-01-03T12:00:00.000000000', ..., '2026-02-07T12:00:00.000000000',
       '2026-02-08T12:00:00.000000000', '2026-02-09T12:00:00.000000000'],
      shape=(31452,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 252kB 1940-01-01T12:00:00 ... 2026-02-09T1...
    crs      float64 8B ...
Attributes:
    standard_name:  time
    axis:           T

In [12]:
tas_nextgems.time

<xarray.DataArray 'time' (time: 10958)> Size: 88kB
[10958 values with dtype=int64]
Dimensions without coordinates: time

In [13]:
tas_nextgems = tas_nextgems.sel(time=slice("2020-01-01", "2020-12-31"))

tas_era5 = tas_era5.sel(time=slice("2020-01-01", "2020-12-31"))

TypeError: 'str' object cannot be interpreted as an integer